# L11c: Markov Decision Processes (MDPs)
A Markov Decision Process extends the basic Markov chain framework by introducing actions and rewards, enabling agents to make optimal decisions in stochastic environments. While standard Markov chains evolve passively according to fixed transition probabilities, MDPs allow an agent to choose actions that influence state transitions and generate rewards, forming the foundation for sequential decision-making under uncertainty.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Formulate a Markov decision process:__ A Markov decision process describes an agent that chooses actions, collects rewards, and moves between states with probabilities that depend on its actions. We'll define its states, actions, rewards, transition model, and discount factor, and the policy that tells the agent which action to take in each state.
> * __Solve for the optimal policy with value iteration:__ Value iteration repeatedly updates the value of each state from the values of the states it can reach next, until the values stop changing. We'll use the converged values to choose the best action in every state.
> * __Estimate values with random rollouts:__ A random rollout simulates the agent taking random actions from a starting state and adds up the discounted rewards it collects. We'll average many rollouts to estimate the value of a state without writing down the transition probabilities.

Let's get started!
___

## MDP Components
Formally, an MDP is defined by the tuple $\left(\mathcal{S}, \mathcal{A}, R\left(s, a\right), T\left(s^{\prime}\,|\,s,a\right), \gamma\right)$:

* **State space** $\mathcal{S}$: The set of all possible states $s$ the system can occupy.
* **Action space** $\mathcal{A}$: The set of all possible actions $a$ available to the agent. For a given state $s$, the available actions are denoted as $\mathcal{A}_{s} \subseteq \mathcal{A}$.
* **Reward function** $R\left(s, a\right)$: The expected immediate reward when taking action $a$ in state $s$. An individual transition produces an observed reward $r$.
* **Transition model** $T\left(s^{\prime}\,|\,s,a\right) = P(s_{t+1} = s^{\prime}\,|\,s_{t}=s,a_{t} = a)$: The probability that action $a$ in state $s$ at time $t$ results in state $s^{\prime}$ at time $t+1$.
* **Discount factor** $\gamma \in [0,1)$: A parameter that weights future rewards relative to immediate rewards, with $\gamma = 0$ prioritizing immediate rewards and $\gamma \to 1$ valuing long-term returns.

The agent's goal is to find a policy $\pi: \mathcal{S} \to \mathcal{A}$ that maps states to actions ($\pi(s) = a$) while maximizing the expected cumulative discounted reward over time, $\mathbb{E}\left[\sum_{t=0}^{\infty}\gamma^{t}R(s_{t},a_{t})\right]$.

For an episodic problem, entering a __terminal state__ ends reward collection. In value iteration, we represent a terminal state as absorbing, with a dummy action that leaves the agent there and earns zero reward. Thus, its value stays zero in every sweep initialized from zero. The reward for reaching the terminal state is collected on entry.

In this lecture, we look at two algorithms: __value iteration__, which solves for the optimal policy, and __random rollout__, which estimates the value of a state by simulation.

### Example: Grid World Navigation

Consider a robot navigating a $3\times 3$ grid world where the goal is to reach a target location while avoiding obstacles and minimizing travel time. This concrete example illustrates how the abstract MDP components work together:

* **State space** $\mathcal{S}$: The eight traversable grid positions, represented as $(x,y)$ coordinates with $x,y\in\{1,2,3\}$, excluding the obstacle at $(2,2)$. The coordinate $x$ increases to the right and $y$ increases upward.
* **Action space** $\mathcal{A}$: The set $\{\text{up}, \text{down}, \text{left}, \text{right}\}$. At boundary positions, actions that would move the agent outside the grid are unavailable, so $\mathcal{A}_s \subsetneq \mathcal{A}$ for edge and corner states.
* **Reward function** $R(s,a)$: A transition into the goal at $(3,3)$ earns $+10$, a bump into the obstacle at $(2,2)$ earns $-5$, and every other transition earns $-1$, including a slip off the grid. These are the full rewards for the respective outcomes.
* **Transition model** $T(s'|s,a)$: Actions are stochastic due to slippery terrain. The intended direction succeeds with probability $0.8$, while the two perpendicular directions each occur with probability $0.1$. For example, choosing "up" results in moving up with probability $0.8$, and moving left or right with probability $0.1$ each.
* **Discount factor** $\gamma = 0.9$: This values the $+10$ goal reward over immediate $-1$ step costs, encouraging the agent to pursue the goal despite accumulated penalties.

Because the moves are random, $R(s,a)$ is the expected reward of action $a$ in state $s$, averaged over where the agent actually moves. A move into the obstacle or off the grid leaves the agent where it is. The goal is terminal, so it earns zero further reward after entry.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
    <img class="course-diagram" src="figs/Fig-GridWorld-Backup.svg" width="740" style="max-width:100%; height:auto;" alt="A three-by-three grid with x increasing rightward and y upward. The center square (2,2) is an obstacle. The goal at (3,3) earns plus ten on entry. The highlighted corner (3,1) has two available actions, up and left. Blue arrows show these intended moves. Bumps into the obstacle earn minus five, and other moves earn minus one."/>
</p>

In this setting, an optimal policy $\pi^{*}$ would map each grid position to the action that maximizes expected cumulative discounted reward, accounting for both the stochastic transitions and the spatial layout of rewards and obstacles.
___

## Value Iteration
Value iteration is a dynamic programming algorithm (it builds the value of each state from the values of the states that can follow it) that computes the optimal value function $U^{*}(s)$ by iteratively applying the Bellman backup operation:

$$
\begin{equation*}
U_{k+1}(s) = \max_{a\in\mathcal{A}_s}\left(R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}}T\left(s^{\prime}\,|\,s,a\right)\cdot{U}_{k}(s^{\prime})\right)
\end{equation*}
$$

For $0\leq\gamma<1$, the value function converges as $k \to \infty$ such that $U_k(s) \to U^{*}(s)$. The optimal value function represents the maximum expected cumulative discounted reward achievable from each state under the best possible policy.


#### Example: One Bellman Backup
Let's update the value of the corner $s=(3,1)$ in the grid above. Its two available actions are up and left. Every possible outcome of either action earns $-1$: a move to a neighboring square or a slip off the grid that leaves the robot at the corner.

Starting from $U_0=0$, the first sweep gives the three values needed for this backup:

$$
U_1(3,2)=7.4,\qquad U_1(2,1)=-1.4,\qquad U_1(3,1)=-1.
$$

For example, the value $7.4$ comes from moving up from $(3,2)$: with probability $0.8$ the robot reaches the goal for $+10$, with probability $0.1$ it bumps the obstacle for $-5$, and with probability $0.1$ it slips off the grid for $-1$.

Let $Q_1(s,a)$ denote the expected return of taking action $a$ and then using the current values $U_1$. Moving up from $(3,1)$ reaches $(3,2)$ with probability $0.8$, slips left to $(2,1)$ with probability $0.1$, and stays at $(3,1)$ with probability $0.1$. Moving left exchanges the probabilities of the two neighboring squares. With $\gamma=0.9$, the two action values are given by:

$$
\begin{aligned}
Q_1((3,1),\mathrm{up})
&=-1+0.9\bigl(0.8(7.4)+0.1(-1.4)+0.1(-1)\bigr)=4.112,\\
Q_1((3,1),\mathrm{left})
&=-1+0.9\bigl(0.8(-1.4)+0.1(7.4)+0.1(-1)\bigr)=-1.432.
\end{aligned}
$$

The Bellman backup sets $U_2(3,1)=\max\{4.112,-1.432\}=4.112$, and the greedy action for these current values is up. Every state in the next sweep uses the values from the previous sweep in the same way.

#### Algorithm
The value iteration algorithm computes the optimal value function $U^{*}(s)$ and policy $\pi^{*}(s)$.

__Initialize__: Given an MDP with state space $\mathcal{S}$, action space $\mathcal{A}$, reward function $R(s,a)$, transition model $T\left(s^{\prime}\,|\,s,a\right)$, discount factor $\gamma$, tolerance $\epsilon>0$, and maximum number of iterations $k_{\max}$, initialize the iteration counter $k\gets 0$, the initial value function $U_{0}(s) \gets 0$ for all $s \in \mathcal{S}$, and $\texttt{converged}\gets\texttt{false}$.

While not $\texttt{converged}$ __do__:
1. For each state $s \in \mathcal{S}$, compute the updated value:
   $$U_{k+1}(s) \gets \max_{a\in\mathcal{A}_s}\left(R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}}T\left(s^{\prime}\,|\,s,a\right)\cdot{U}_{k}(s^{\prime})\right)$$
2. Check whether to stop:
    - If $\max_{s\in\mathcal{S}} \left|U_{k+1}(s) - U_{k}(s)\right| \leq \epsilon$, then set $\texttt{converged}\gets\texttt{true}$ and return $U^{*}\gets{U}_{k+1}$.
    - Otherwise, if $k+1\geq{k_{\max}}$, then return $U_{k+1}$ with $\texttt{converged}=\texttt{false}$ and a warning that the tolerance was not met.
3. Increment the iteration counter: $k\gets{k+1}$.

__Extract Policy__: For each state $s \in \mathcal{S}$, compute:
$$\pi^{*}(s) \gets \arg\max_{a\in\mathcal{A}_s}\left(R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}}T\left(s^{\prime}\,|\,s,a\right)\cdot{U^{*}}(s^{\prime})\right)$$

#### Computational Complexity

Value iteration requires $O(|\mathcal{S}|^2 \cdot |\mathcal{A}|)$ operations per iteration. For each of the $|\mathcal{S}|$ states, we must evaluate $|\mathcal{A}|$ actions, and each action evaluation requires summing over $|\mathcal{S}|$ possible next states. The number of iterations until convergence depends on the discount factor $\gamma$ and the tolerance $\epsilon$, because each iteration shrinks the largest error in the value function by at least a factor of $\gamma$.

This complexity makes value iteration tractable for problems with thousands of states but challenging for continuous or very large discrete state spaces. In such cases, function approximation or sampling-based methods become necessary.

___

## Random Rollout Algorithm

The random rollout algorithm estimates state values by simulating random trajectories from a starting state and computing the cumulative discounted reward obtained along each path. Unlike value iteration, which requires knowledge of the transition model $T(s^{\prime}\,|\,s,a)$ and computes values for all states simultaneously, random rollout is a model-free sampling approach that explores the state space through direct interaction with the environment.

The algorithm generates trajectories by selecting random actions at each state, executing them in the environment, and accumulating the observed discounted rewards. By averaging the returns from multiple rollouts starting from a given state $s$, we obtain an empirical estimate of the value function $\hat{U}(s)$.

#### Algorithm

The random rollout algorithm estimates the value function $\hat{U}(s)$ through Monte Carlo sampling:

__Initialize__: Given an environment that can simulate actions and report next states and rewards, its state space $\mathcal{S}$ and available actions $\mathcal{A}_s$, a discount factor $0\leq\gamma<1$, a maximum depth $d\geq 1$, and a number of rollouts $N\geq 1$, initialize the value estimates $\hat{U}(s) \gets 0$ and visit counts $n(s) \gets 0$ for all $s \in \mathcal{S}$.

For $i = 1$ to $N$ __do__:

1. Initialize the rollout:
    - Set starting state $s_{0}$ (chosen uniformly or from a start distribution).
    - Set depth counter $t\gets 0$ and cumulative return $G\gets 0$.
    - Set current state $s\gets s_0$. Set $\texttt{terminated}\gets\texttt{true}$ if $s_0$ is terminal; otherwise set it to $\texttt{false}$.

2. While $\texttt{terminated}$ is $\texttt{false}$ __do__:
    - Select action $a$ uniformly at random from $\mathcal{A}_{s}$.
    - Execute action $a$ and observe its reward $r$ and next state $s^{\prime}$.
    - Update cumulative return: $G\gets G+\gamma^t r$.
    - Update depth: $t \gets t + 1$.
    - Check termination conditions:
        - If $t \geq d$ (maximum depth reached), set $\texttt{terminated}\gets\texttt{true}$.
        - If $s^{\prime}$ is a terminal state, set $\texttt{terminated}\gets\texttt{true}$.
        - Otherwise, update $s\gets s^{\prime}$.

3. Update value estimates:
    - Increment visit count: $n(s_{0}) \gets n(s_{0}) + 1$.
    - Update value estimate using incremental mean:
       $$\hat{U}(s_{0}) \gets \hat{U}(s_{0}) + \frac{1}{n(s_{0})}\left(G - \hat{U}(s_{0})\right)$$

__Output__: The estimated value $\hat{U}(s)$ for each state $s$ that served as a starting state $s_{0}$ in the $N$ rollouts.

#### Convergence Properties

As the number of independent rollouts from a starting state $s_0$ grows, its average return converges to the expected discounted return of a rollout that follows the __random policy__, the policy that picks each available action with equal probability, and stops after $d$ steps or on reaching a terminal state. The estimate describes the random policy with this depth limit; value iteration computes the optimal value $U^{*}(s_0)$. Revisited states remain part of the trajectory, so a bump or a return to an earlier square still allows the robot to continue.

For bounded observed rewards $|r|\leq r_{\max}$, where $r_{\max}$ is the largest possible reward magnitude, the omitted discounted return after $d$ steps is bounded by:

$$
\left|U^{\mathrm{random}}(s)-U_d^{\mathrm{random}}(s)\right|
\leq\sum_{t=d}^{\infty}\gamma^t r_{\max}
=\frac{\gamma^d r_{\max}}{1-\gamma}.
$$

Here, $U^{\mathrm{random}}$ is the random policy's expected return without the depth limit, and $U_d^{\mathrm{random}}$ is its expected return with the limit. Increasing the depth reduces this truncation error; increasing the number of rollouts reduces sampling uncertainty. Neither the transition probabilities nor the expected reward table is needed when the environment supplies next states and observed rewards.

#### Computational Complexity

Random rollout requires $O(N \cdot d)$ simulated steps, where $N$ is the number of rollouts and $d$ is the maximum depth per rollout, assuming that choosing a random action and simulating it takes constant time. Apart from storing one estimate per state, the cost does not depend on the state space size $|\mathcal{S}|$, making random rollout particularly attractive for large or continuous state spaces.

For independent rollouts from a given starting state $s$, the variance of its sample mean decreases as $O(1/n(s))$, so its standard error decreases as $O(1/\sqrt{n(s)})$, meaning accuracy improves slowly with additional samples. However, the algorithm can provide useful estimates with relatively few rollouts and scales naturally to high-dimensional problems where model-based approaches become intractable.
___

## Lab
In lab `L11d`, we build a grid world in which a robot must reach a charging station while avoiding lava pits, write our own value iteration with a convergence check, and use the optimal policy to plan the robot's path.

___

## Summary
In this lecture, we explored how Markov decision processes extend Markov chains by adding actions and rewards to enable optimal sequential decision-making:

> __Key Takeaways:__
>
> * __MDPs add actions and rewards to Markov chains:__ We added actions that change the transition probabilities and rewards that score each choice, so the agent's decisions now shape where the process goes. The discount factor sets how much the agent values future rewards against immediate ones, and a policy assigns an action to every state.
>
> * __Value iteration finds the optimal policy:__ We applied the Bellman backup to every state until the values stopped changing, which gives the maximum expected discounted reward from each state when the discount factor is less than one. Choosing the action that achieves that maximum in each state gives the optimal policy.
>
> * __Random rollout estimates values by simulation:__ We simulated random trajectories until termination or a depth limit and averaged their observed discounted rewards. The depth limit controls the omitted future return, while the number of rollouts controls sampling uncertainty. This needs only a way to simulate the environment, not its transition probabilities, and it tells us how well random actions do, not the optimal value.

Next week, the agent no longer knows the transition model and learns the value of each action from its own experience with Q-learning.

___